# Bank Marketing Term Deposit Prediction

**Tabular Classification Project**

## 2 · Project Overview

Predict whether a bank customer will subscribe to a **term deposit** based on demographics, economic indicators, and marketing campaign data. Synthetic version with ~4,500 rows and ~47% positive rate.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given customer demographics, account information, and marketing campaign details, predict whether the customer will subscribe to a term deposit (deposit = 1).

## 5 · Why This Project Matters

- Banking marketing campaigns cost millions — targeting the right customers is critical.
- Predicting subscription likelihood enables **personalized outreach** and resource optimization.
- This dataset teaches handling of **mixed feature types** and marketing attribution.
- Marketing ML is one of the largest enterprise ML application areas.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~4,500 |
| Features | 16 (age, job, marital, education, balance, housing, loan, contact, day, month, duration, campaign, pdays, previous, poutcome) |
| Target | `deposit` (binary: 0=no, 1=yes) |
| Class balance | ~53% no, ~47% yes |
| Missing values | None (categoricals encode 'unknown') |

## 7 · Dataset Source and License Notes

**Source**: UCI ML Repository — Bank Marketing dataset.
**License**: Public / educational.
**Citation**: Moro, Cortez & Rita, 2014.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "deposit"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Bank Marketing Term Deposit Prediction


## 11 · Dataset Download or Loading

In [4]:
np.random.seed(SEED)
n = 4521

age = np.random.normal(41, 11, n).clip(18, 95).astype(int)
job = np.random.choice(['admin.', 'blue-collar', 'entrepreneur', 'housemaid', 'management',
                        'retired', 'self-employed', 'services', 'student', 'technician', 'unemployed', 'unknown'], n)
marital = np.random.choice(['married', 'single', 'divorced'], n, p=[0.6, 0.28, 0.12])
education = np.random.choice(['primary', 'secondary', 'tertiary', 'unknown'], n, p=[0.15, 0.5, 0.3, 0.05])
balance = np.random.lognormal(7, 1.5, n).astype(int) - 500
balance = balance.clip(-2000, 80000)
housing = np.random.choice(['yes', 'no'], n, p=[0.55, 0.45])
loan = np.random.choice(['yes', 'no'], n, p=[0.16, 0.84])
contact = np.random.choice(['cellular', 'telephone', 'unknown'], n, p=[0.6, 0.25, 0.15])
day = np.random.randint(1, 31, n)
month = np.random.choice(['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec'], n)
duration = np.random.lognormal(5.5, 0.8, n).clip(2, 4000).astype(int)
campaign = np.random.poisson(2.5, n).clip(1, 50)
pdays = np.where(np.random.random(n) < 0.6, -1, np.random.randint(1, 400, n))
previous = np.where(pdays == -1, 0, np.random.poisson(1.5, n).clip(0, 25))
poutcome = np.where(pdays == -1, 'unknown',
                    np.random.choice(['success', 'failure', 'other'], n, p=[0.3, 0.5, 0.2]))

score = (
    0.002 * duration
    + 0.3 * (poutcome == 'success').astype(float)
    - 0.2 * (housing == 'yes').astype(float)
    + 0.1 * (education == 'tertiary').astype(float)
    - 0.01 * campaign
    + 0.15 * (contact == 'cellular').astype(float)
    + np.random.normal(0, 0.5, n)
)
deposit = (score > 0.6).astype(int)

df = pd.DataFrame({
    'age': age, 'job': job, 'marital': marital, 'education': education,
    'balance': balance, 'housing': housing, 'loan': loan, 'contact': contact,
    'day': day, 'month': month, 'duration': duration, 'campaign': campaign,
    'pdays': pdays, 'previous': previous, 'poutcome': poutcome, 'deposit': deposit,
})
print(f"Dataset shape: {df.shape}")
print(f"Deposit rate: {df['deposit'].mean():.2%}")
df.head()

Dataset shape: (4521, 16)
Deposit rate: 48.91%


,age,job,marital,education,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,46,student,married,tertiary,4994,no,no,cellular,3,apr,411,3,123,0,success,0
1,39,blue-collar,married,secondary,3045,no,no,unknown,29,sep,658,2,-1,0,unknown,1
2,48,retired,married,primary,896,no,yes,cellular,10,may,63,1,174,0,other,0
3,57,blue-collar,single,tertiary,1359,no,no,telephone,12,apr,261,4,71,1,failure,1
4,38,management,married,secondary,-405,yes,no,cellular,23,sep,57,3,3,2,other,0


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (4521, 16)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 0

Target distribution:
deposit
0    2310
1    2211
Name: count, dtype: int64

Dtypes:
age           int64
job          object
marital      object
education    object
balance       int64
housing      object
loan         object
contact      object
day           int32
month        object
duration      int64
campaign      int32
pdays         int32
previous      int32
poutcome     object
deposit       int64
dtype: object

Target 'deposit' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(num_cols[:6]):
    ax = axes[i // 3, i % 3]
    df[col].hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(col, fontsize=10)
plt.suptitle("Numeric Feature Distributions", fontsize=14)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(['job', 'education', 'poutcome']):
    ct = pd.crosstab(df[col], df[TARGET], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=['steelblue', 'coral'])
    axes[i].set_title(f"Deposit Rate by {col}")
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()
print(f"Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")

Numeric: 7, Categorical: 8


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title("Term Deposit Subscription")
axes[0].set_xticklabels(['No (0)', 'Yes (1)'], rotation=0)
df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'coral'])
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts()}")

Class distribution:
deposit
0    2310
1    2211
Name: count, dtype: int64


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[cat_cols] = oe.fit_transform(df[cat_cols])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

Train: (3616, 15), Test: (905, 15)
Train target dist: {0: 1848, 1: 1768}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (15): ['age', 'job', 'marital', 'education', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome']


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
y_prob_bl = baseline.predict_proba(X_test)[:, 1] if len(np.unique(y_train)) == 2 else None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.7392
Precision: 0.7439
Recall   : 0.7392
F1       : 0.7373
ROC-AUC  : 0.8024


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                            Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                       
CalibratedClassifierCV      0.752486           0.750613  0.808415  0.750280   0.758967  0.752486    0.055319
LogisticRegression          0.746961           0.745016  0.808014  0.744541   0.753824  0.746961    0.020582
LinearSVC                   0.745856           0.743748  0.808361  0.743016   0.754090  0.745856    0.021059
AdaBoostClassifier          0.744751           0.742248  0.795384  0.740763   0.756968  0.744751    0.194543
BernoulliNB                 0.743646           0.740980  0.776025  0.739113   0.757718  0.743646    0.018700
CatBoostClassifier          0.742541           0.740687  0.805991  0.740301   0.748517  0.742541    2.363289
SVC                         0.731492           0.729168  0.796439  0.727884   0.741020  0.7314

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: lgbm
Accuracy : 0.7470
F1       : 0.7447


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.7481  F1: 0.7456


LightGBM  Acc: 0.7050  F1: 0.7038


XGBoost   Acc: 0.7039  F1: 0.7027


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
CatBoost               0.7481  0.7456     0.7552  0.7481
FLAML                  0.7470  0.7447     0.7534  0.7470
Logistic Regression    0.7392  0.7373     0.7439  0.7392
LightGBM               0.7050  0.7038     0.7066  0.7050
XGBoost                0.7039  0.7027     0.7055  0.7039

Top 3: ['CatBoost', 'FLAML', 'Logistic Regression']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  CatBoost
              precision    recall  f1-score   support

           0       0.72      0.84      0.77       462
           1       0.80      0.65      0.72       443

    accuracy                           0.75       905
   macro avg       0.76      0.75      0.75       905
weighted avg       0.76      0.75      0.75       905


  FLAML
              precision    recall  f1-score   support

           0       0.72      0.84      0.77       462
           1       0.79      0.65      0.72       443

    accuracy                           0.75       905
   macro avg       0.75      0.75      0.74       905
weighted avg       0.75      0.75      0.74       905


  Logistic Regression
              precision    recall  f1-score   support

           0       0.71      0.82      0.76       462
           1       0.78      0.66      0.71       443

    accuracy                           0.74       905
   macro avg       0.74      0.74      0.7

## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: CatBoost
Error rate: 0.2519 (228 / 905)

Errors by true class:
      errors  total  error_rate
true                           
0         74    462    0.160173
1        154    443    0.347630


## 25 · Interpretation and Business Insight

- **Call duration** is the strongest predictor — longer calls indicate interest.
- **Previous campaign outcome** (success) strongly predicts future subscription.
- **Housing loan** reduces deposit likelihood (less savings).
- **Education** (tertiary) slightly increases subscription.
- **Duration is a leakage risk** — it's only known after the call ends.

## 26 · Limitations

1. **Duration leakage** — call duration is known post-call, not pre-call.
2. Synthetic data — real marketing depends on timing, offers, and market conditions.
3. No cost-benefit analysis for campaign ROI.
4. 'Unknown' categories are informative but not true missing values.
5. Monthly patterns may reflect campaign scheduling, not customer behavior.

## 27 · How to Improve This Project

1. Remove duration (leakage) and evaluate — the true benchmark.
2. Engineer contact recency and frequency features.
3. Build a campaign ROI model (cost per conversion).
4. Add macroeconomic indicators (interest rates, GDP).
5. Use uplift modeling to find persuadable customers.

## 28 · Production Considerations

- Real-time call scoring during marketing campaigns.
- Integration with CRM for lead prioritization.
- Campaign optimization and budget allocation.
- A/B testing framework for outreach strategies.

## 29 · Common Mistakes

1. Including duration as a feature (it's post-hoc leakage).
2. Not computing per-campaign ROI.
3. Treating 'unknown' as missing instead of a distinct category.
4. Optimizing accuracy instead of conversion rate.
5. Not segmenting customers by lifetime value.

## 30 · Mini Challenge / Exercises

1. Remove 'duration' and retrain — how much does F1 drop?
2. Calculate the cost per conversion for different thresholds.
3. Segment by job type and compare conversion rates.
4. Build a model that predicts which 20% of customers to call first.

## 31 · Final Summary / Key Takeaways

1. Bank marketing prediction teaches real-world ML deployment challenges.
2. Call duration is a leakage risk — always question feature availability at prediction time.
3. Previous campaign outcome is a strong predictor of future behavior.
4. Marketing ML requires cost-benefit analysis, not just accuracy.
5. Uplift modeling (who to target) is more valuable than response modeling.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.7481,
    "f1": 0.7456,
    "precision": 0.7552,
    "recall": 0.7481
  },
  "LightGBM": {
    "accuracy": 0.705,
    "f1": 0.7038,
    "precision": 0.7066,
    "recall": 0.705
  },
  "XGBoost": {
    "accuracy": 0.7039,
    "f1": 0.7027,
    "precision": 0.7055,
    "recall": 0.7039
  },
  "Logistic Regression": {
    "accuracy": 0.7392,
    "f1": 0.7373,
    "precision": 0.7439,
    "recall": 0.7392
  },
  "FLAML": {
    "accuracy": 0.747,
    "f1": 0.7447,
    "precision": 0.7534,
    "recall": 0.747
  }
}
